In [2]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime as dt

Set parameters for the company size and date ranges:

In [ ]:
n_employees = 100
n_projects = 40
n_clients = 15
n_weeks = 24
start_date = dt(2025, 1, 1)
end_date = dt(2026, 12, 31)
bench = 0.15 
working_hours = 8

In [4]:
job_matrix_df = pd.DataFrame(
    {
            "Role": ["Project Manager", "Business Analyst", "Software Engineer", "Tester", "Solution Architect", "DevOps Engineer", "Platform Engineer", "Cloud Engineer"],
            "Cost_Rate": [55, 50, 75, 45, 100, 60, 55, 30]
    }
)

job_matrix_df

,Role,Cost_Rate
0,Project Manager,55
1,Business Analyst,50
2,Software Engineer,75
3,Tester,45
4,Solution Architect,100
5,DevOps Engineer,60
6,Platform Engineer,55
7,Cloud Engineer,30


In [5]:
#Define the department dataset
department_df = pd.DataFrame(
            {
            "Department_ID": ["D01", "D02", "D03", "D04"],
            "Department_Name": ["Energy", "Finance", "Healthcare", "Retail"],}
        )
department_df

,Department_ID,Department_Name
0,D01,Energy
1,D02,Finance
2,D03,Healthcare
3,D04,Retail


In [6]:
#Define the resource dataset

rng = np.random.default_rng(42)
fake = Faker()
fake.seed_instance(42)

resource_df = pd.DataFrame(
        {
        "Employee_ID": [f"E{i}" for i in range(1, n_employees + 1)],
        "Department_ID": rng.choice(department_df["Department_ID"], n_employees),
        "Role": rng.choice(job_matrix_df["Role"], n_employees),
        "Level": rng.choice([1, 2, 3], n_employees),
        "Name": [fake.name() for _ in range(n_employees)]
        }
    )

resource_df

,Employee_ID,Department_ID,Role,Level,Name
0,E1,D01,Platform Engineer,2,Allison Hill
1,E2,D04,Business Analyst,3,Noah Rhodes
2,E3,D03,Platform Engineer,2,Angie Henderson
3,E4,D02,Project Manager,3,Daniel Wagner
4,E5,D02,Platform Engineer,2,Cristian Santos
...,...,...,...,...,...
95,E96,D02,Project Manager,2,Anna Henderson
96,E97,D01,Software Engineer,3,Aaron Wise
97,E98,D03,Project Manager,2,Deborah Figueroa
98,E99,D03,Software Engineer,1,Jessica Smith


In [7]:
#Define the project dataset
project_df = pd.DataFrame(
        {
        "Project_ID" : [f"P{i}" for i in range(1, n_projects + 1)],
        "Project_Name": [fake.bs().title() for _ in range(n_projects)],
        "Project_Type": rng.choice(["Fixed Price", "T&M"], n_projects),
        "Client ID": rng.choice([f"C{i}" for i in range(1, n_clients + 1)], n_projects),
        "Department_ID": rng.choice(department_df["Department_ID"], n_projects),
        "Start_Date": [fake.date_between(start_date, end_date) for _ in range(n_projects)],
        "Duration": rng.choice([3, 6, 9, 12, 18, 24], n_projects),
        "Project_Manager_ID": rng.choice(resource_df[resource_df["Role"] == "Project Manager"]["Employee_ID"], n_projects),
        "Contract Value": [fake.pydecimal(left_digits=6, right_digits=2, positive=True) for _ in range(n_projects)]}
    )

project_df

,Project_ID,Project_Name,Project_Type,Client ID,Department_ID,Start_Date,Duration,Project_Manager_ID,Contract Value
0,P1,Incubate Robust Relationships,Fixed Price,C10,D01,2025-08-11,24,E71,356736.79
1,P2,Benchmark B2B E-Commerce,T&M,C5,D04,2025-06-13,18,E17,991014.17
2,P3,Enable Revolutionary Content,Fixed Price,C10,D04,2025-11-30,18,E71,534572.93
3,P4,Disintermediate Revolutionary Convergence,T&M,C8,D01,2026-09-03,12,E38,853240.67
4,P5,Architect Interactive Web Services,T&M,C14,D01,2026-03-27,9,E4,519066.32
5,P6,Transform Synergistic Schemas,Fixed Price,C4,D01,2026-04-17,6,E98,787926.01
6,P7,Matrix Transparent E-Business,Fixed Price,C15,D01,2026-05-24,24,E20,916232.63
7,P8,Extend B2C Action-Items,T&M,C15,D04,2025-09-08,18,E50,284430.87
8,P9,Cultivate Granular Roi,Fixed Price,C3,D04,2025-12-28,3,E20,152657.83
9,P10,Matrix Transparent Web-Readiness,Fixed Price,C3,D01,2025-12-10,18,E98,989545.17


In [ ]:
#Assign people to projects, based on bench parameter, assumed 100% utilised and 1 project per employee
assignment_df = pd.DataFrame(
        {
        "Project_ID": rng.choice(project_df["Project_ID"], int(n_employees*(1 - bench))),
        "Employee_ID": rng.choice(resource_df["Employee_ID"], int(n_employees*(1 - bench)), replace=False),
        "Assignment_Start_Date": [fake.date_between(start_date, end_date) for _ in range(int(n_employees*(1 - bench)))],
        "Assignment_End_Date": }
    )

assignment_df


,Project_ID,Employee_ID,Assignment_Start_Date,Assignment_End_Date
0,P30,E93,2026-11-05,2026-10-23
1,P24,E71,2025-12-25,2026-10-23
2,P3,E39,2025-05-22,2026-10-23
3,P16,E78,2025-02-09,2026-10-23
4,P20,E59,2025-07-22,2026-10-23
...,...,...,...,...
80,P11,E23,2025-07-08,2026-10-23
81,P32,E74,2025-03-21,2026-10-23
82,P19,E87,2026-06-26,2026-10-23
83,P8,E38,2026-08-24,2026-10-23


In [ ]:
#Define the timesheet dataset

timesheet_df = pd.DataFrame(
        {
        "Employee_ID": [],
        "Project_ID": [],
        "Period_Start_Date": [],
        "Billable_Hours": [],
        "Internal_Hours": []}
    )

project_df